In [ ]:
%matplotlib widget

import logging
import httpx
import matplotlib.pyplot as plt
import pandas as pd

from arpav_ppcv.webapp.api_v2.schemas import (
    coverages,
    timeseries,
)

logging.basicConfig(level=logging.DEBUG)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("matplotlib.pyplot").setLevel(logging.WARNING)

base_url = "http://webapp:5001/api/v2"
# location = "POINT(11.4134 45.7684)"
location = "POINT(12.3003 45.5826)"


client = httpx.Client()

cov_identifier = "historical-tas-absolute-annual-arpa_v-only_year-all_year"

def to_pandas_series(series: timeseries.LegacyTimeSeries) -> pd.Series:
    return pd.Series(
        data=[v.value for v in series.values],
        index=[v.datetime for v in series.values],
        name=series.name
    )


time_series_response = client.get(
    f"{base_url}/coverages/historical-time-series/{cov_identifier}",
    params={
        "coords": location,
        "datetime": "../..",
        "include_coverage_data": True,
        "include_observation_data": True,
        "coverage_processing_methods": [
            "no_processing",
            "loess_smoothing",
            "moving_average_11_years",
            "decade_aggregation",
            "mann_kendall_trend",
        ],
        "observation_processing_methods": [
            "no_processing",
            "moving_average_5_years",
        ],
    }
)
try:
    time_series_response.raise_for_status()
except httpx.HTTPStatusError as err:
    print(time_series_response.json())
    raise
parsed = time_series_response.json()

cov_series = []

for dict_series in parsed["series"]:
    time_series = timeseries.LegacyTimeSeries(**dict_series)
    print(time_series.name)
    cov_series.append(time_series)

In [ ]:
fig2, ax2 = plt.subplots()
for s in cov_series:
    pd_series = to_pandas_series(s)
    if "decade" in pd_series.name:
        pd_series.plot(ax=ax2, drawstyle="steps")
    else:
        pd_series.plot(ax=ax2)

fig2.legend()
fig2.tight_layout()

In [ ]:
cov_series[-1].info

In [ ]:
cov_series[-2].info

In [ ]:
cov_series[-3].info